In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# 1. CONSTANTS & PHYSICAL PARAMETERS
# =============================================================================
EPSILON_0 = 8.854e-12
C_LIGHT = 3e8
LAMBDA_0 = 1294.0e-9           # Target wavelength in meters (1294 nm)
OMEGA_0 = 2 * np.pi * C_LIGHT / LAMBDA_0
maj_tick_width = 0.8
min_tick_width = 0.8

plot_params = {
    "figure.dpi": 200,
    "axes.labelsize": 14,
    "axes.linewidth": 1.5,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.title_fontsize": 11,
    "legend.fontsize": 11,
    "xtick.major.size": 3.5,
    "xtick.major.width": maj_tick_width,
    "xtick.minor.size": 2.5,
    "xtick.minor.width": min_tick_width,
    "ytick.major.size": 3.5,
    "ytick.major.width": maj_tick_width,
    "ytick.minor.size": 2.5,
    "ytick.minor.width": min_tick_width,
    "font.family": "serif",
    "text.usetex": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pgf.texsystem": "pdflatex",
    "pgf.rcfonts": False,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
}

plt.rcParams.update(plot_params)
# Default simulation parameters matching your blueprint setup
PARAMS = {
    'q': 1.602e-19,             # Elementary charge
    'eta': 3.4,                 # Refractive index of background material
    'N_l': 5.0                  # Number of QD layers
}

def refractive_index_change(n_e_sch, n_e_w, n_e_im, n_h_sch, n_h_qd,
                           m_e=0.0465, m_h=0.455,
                           gamma_xy=0.088, gamma_xy_sch=0.06985092,
                           h_w=5e-9, h_sch=430e-9, dot=True,
                           S_re_e=0.8, S_re_h=0.8,
                           params=PARAMS, omega_0=OMEGA_0):
    """
    Computes plasma-induced refractive index change using the Uskov formulation.
    """
    coeff = -params['q']**2 / (2 * params['eta'] * EPSILON_0 * omega_0**2) 

    if dot:
        effective_electrons = n_e_w + (S_re_e * np.sum(n_e_im))
        electron_term = (gamma_xy / (0.064 * 9.11e-31 * h_w * params['N_l'])) * effective_electrons
        hole_term = (gamma_xy / (0.5 * 9.11e-31 * h_w * params['N_l'])) * (S_re_h * n_h_qd)
        
        electron_term_sch = gamma_xy_sch * n_e_sch / (0.064 * 9.11e-31 * h_sch)
        hole_term_sch = gamma_xy_sch * n_h_sch / (0.5 * 9.11e-31 * h_sch)
        
    return coeff * (electron_term + hole_term + electron_term_sch + hole_term_sch)


# =============================================================================
# 2. LOAD DATA & SWEEP ALPHA FACTOR
# =============================================================================
def calculate_alpha_vs_current(data_filepath):
    """
    Reads tabular data containing base and perturbed state calculations,
    evaluates dg/dN, dn/dN, and computes alpha_H vs. current.
    Uses electron carrier density N in units of 1e18 cm^-3.
    """
    df = pd.read_csv(data_filepath)

    unique_currents = df['nominal_current_mA'].unique()
    
    alpha_list = []
    dgdN_list = []
    dndN_list = []

    for I in unique_currents:
        base_rows = df[(df['nominal_current_mA'] == I) & (df['modulation'] == 'base')]
        plus_rows = df[(df['nominal_current_mA'] == I) & (df['modulation'] == 'plus1pct')]

        if len(base_rows) == 0 or len(plus_rows) == 0:
            continue

        base = base_rows.iloc[0]
        plus = plus_rows.iloc[0]

        # 1. Calculate electron carrier density N (in units of 1e18 cm^-3)
        n_im_base = np.array([base['gs'], base['es1'], base['es2']]) * 1e18
        n_im_plus = np.array([plus['gs'], plus['es1'], plus['es2']]) * 1e18

        # Electron density in units of 1e18 cm^-3 (do NOT multiply the total N by 1e18)
        N_base = base['sch'] + base['wl'] + base['gs'] + base['es1'] + base['es2']
        N_plus = plus['sch'] + plus['wl'] + plus['gs'] + plus['es1'] + plus['es2']
        dN = N_plus - N_base

        # 2. Refractive index calculation (requires 1e18 scaling for Uskov model)
        n_ref_base = refractive_index_change(
            n_e_sch=base['sch'] * 1e18, n_e_w=base['wl'] * 1e18, n_e_im=n_im_base,
            n_h_sch=base['sch'] * 1e18, n_h_qd=(np.sum(n_im_base) + base['wl'] * 1e18)
        )
        n_ref_plus = refractive_index_change(
            n_e_sch=plus['sch'] * 1e18, n_e_w=plus['wl'] * 1e18, n_e_im=n_im_plus,
            n_h_sch=plus['sch'] * 1e18, n_h_qd=(np.sum(n_im_plus) + plus['wl'] * 1e18)
        )

        dn = n_ref_plus - n_ref_base

        # 3. Differential gain calculation (dg/dN)
        dg = plus['gain'] - base['gain']
        dgdN = dg / dN

        # 4. Linewidth enhancement factor (alpha_H)
        # Note: dN cancels out in dn/dg = (dn/dN)/(dg/dN), so alpha_H is independent of scaling.
        lam_cm = LAMBDA_0 * 100.0  # Convert wavelength from meters to cm
        alpha_H = -4.0 * np.pi / lam_cm * (dn / dg)

        alpha_list.append(alpha_H)
        dgdN_list.append(dgdN)
        dndN_list.append(dn / dN)

    return np.array(unique_currents), np.array(alpha_list), np.array(dgdN_list), np.array(dndN_list)


# =============================================================================
# 3. PLOTTING
# =============================================================================
if __name__ == '__main__':
    linear_path = 'alpha_sweep_summary_linear.csv'
    optimum_path = 'alpha_sweep_summary_optimum.csv'

    curr_lin, alpha_lin, dgdN_lin, dndN_lin = calculate_alpha_vs_current(linear_path)
    curr_opt, alpha_opt, dgdN_opt, dndN_opt = calculate_alpha_vs_current(optimum_path)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: dg/dN vs Current
    axes[0].plot(curr_lin, dgdN_lin, 'o-', color='tab:blue', label='Linear Run')
    axes[0].plot(curr_opt, dgdN_opt, 's-', color='tab:orange', label='Optimum Run')
    axes[0].set_xlabel('Current (mA)')
    axes[0].set_ylabel(r'd$g$/d$N$ ($\mathrm{cm}^{-1} / (10^{18} \mathrm{cm}^{-3})$)')
    axes[0].set_title('Differential Gain')
    axes[0].grid(True, linestyle='--', alpha=0.5)
    axes[0].legend()

    # Plot 2: dn/dN vs Current
    axes[1].plot(curr_lin, dndN_lin, 'o-', color='tab:purple', label='Linear Run')
    axes[1].plot(curr_opt, dndN_opt, 's-', color='tab:pink', label='Optimum Run')
    axes[1].set_xlabel('Current (mA)')
    axes[1].set_ylabel(r'd$n$/d$N$ ($(10^{18} \mathrm{cm}^{-3})^{-1}$)')
    axes[1].set_title('Refractive Index Derivative')
    axes[1].grid(True, linestyle='--', alpha=0.5)
    axes[1].legend()

    # Plot 3: Alpha Factor vs Current
    axes[2].plot(curr_lin, alpha_lin, 'o-', color='tab:red', label='Linear Run')
    axes[2].plot(curr_opt, alpha_opt, 's-', color='tab:green', label='Optimum Run')
    axes[2].set_xlabel('Current (mA)')
    axes[2].set_ylabel(r'$lpha_H$')
    axes[2].set_title('Linewidth Enhancement Factor')
    axes[2].grid(True, linestyle='--', alpha=0.5)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

RuntimeError: latex was not able to process the following string:
b'$\\x07lpha_H$'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2025-10-24>
(c:/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(c:/texlive/2025/texmf-dist/tex/latex/base/size10.clo))
(c:/texlive/2025/texmf-dist/tex/latex/type1cm/type1cm.sty)
(c:/texlive/2025/texmf-dist/tex/latex/cm-super/type1ec.sty
(c:/texlive/2025/texmf-dist/tex/latex/base/t1cmr.fd))
(c:/texlive/2025/texmf-dist/tex/latex/base/inputenc.sty)
(c:/texlive/2025/texmf-dist/tex/latex/geometry/geometry.sty
(c:/texlive/2025/texmf-dist/tex/latex/graphics/keyval.sty)
(c:/texlive/2025/texmf-dist/tex/generic/iftex/ifvtex

Error in callback <function _draw_all_if_interactive at 0x0000024AFA6200E0> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: latex was not able to process the following string:
b'$\\x07lpha_H$'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2025-10-24>
(c:/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(c:/texlive/2025/texmf-dist/tex/latex/base/size10.clo))
(c:/texlive/2025/texmf-dist/tex/latex/type1cm/type1cm.sty)
(c:/texlive/2025/texmf-dist/tex/latex/cm-super/type1ec.sty
(c:/texlive/2025/texmf-dist/tex/latex/base/t1cmr.fd))
(c:/texlive/2025/texmf-dist/tex/latex/base/inputenc.sty)
(c:/texlive/2025/texmf-dist/tex/latex/geometry/geometry.sty
(c:/texlive/2025/texmf-dist/tex/latex/graphics/keyval.sty)
(c:/texlive/2025/texmf-dist/tex/generic/iftex/ifvtex

RuntimeError: latex was not able to process the following string:
b'$\\x07lpha_H$'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2025-10-24>
(c:/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(c:/texlive/2025/texmf-dist/tex/latex/base/size10.clo))
(c:/texlive/2025/texmf-dist/tex/latex/type1cm/type1cm.sty)
(c:/texlive/2025/texmf-dist/tex/latex/cm-super/type1ec.sty
(c:/texlive/2025/texmf-dist/tex/latex/base/t1cmr.fd))
(c:/texlive/2025/texmf-dist/tex/latex/base/inputenc.sty)
(c:/texlive/2025/texmf-dist/tex/latex/geometry/geometry.sty
(c:/texlive/2025/texmf-dist/tex/latex/graphics/keyval.sty)
(c:/texlive/2025/texmf-dist/tex/generic/iftex/ifvtex

<Figure size 3600x1000 with 3 Axes>